<a href="https://colab.research.google.com/github/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised/blob/main/python/full_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
## Setup and Data Acquisition
First, we clone the repository and install the necessary Python dependencies.
```

In [19]:
import os
# Remove old clone if it exists to ensure a fresh start
!rm -rf Semantically_Incongruent_or_Congruent_Eggplants_revised

# Clone the updated repository
!git clone https://github.com/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised.git

# Install dependencies from the python folder
%cd Semantically_Incongruent_or_Congruent_Eggplants_revised/python
!pip install -r requirements.txt
%cd ../..

Cloning into 'Semantically_Incongruent_or_Congruent_Eggplants_revised'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 38 (delta 5), reused 29 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 90.68 MiB | 46.78 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/Semantically_Incongruent_or_Congruent_Eggplants_revised/python
/content


```markdown
Next, we check the stimuli folder and unzip it. Note: We use a manual download and pushed to GitHub, Dryad API did not work for automated request.
```

In [20]:
import os
import zipfile

base_dir = 'Semantically_Incongruent_or_Congruent_Eggplants_revised'
stimuli_path = os.path.join(base_dir, 'stimuli')
dataset_path = 'dataset'
os.makedirs(dataset_path, exist_ok=True)

if not os.path.exists(stimuli_path):
    print(f'Error: {stimuli_path} not found. Listing current directory:')
    !ls -R {base_dir}
else:
    files = [f for f in os.listdir(stimuli_path) if f.endswith('.zip')]
    print(f'Found {len(files)} zip files in {stimuli_path}')
    for file in files:
        file_path = os.path.join(stimuli_path, file)
        print(f'Extracting {file}...')
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            zip_ref.extractall(dataset_path)

    print(f'\nExtraction complete. Files in {dataset_path}:')
    print(os.listdir(dataset_path))

Found 1 zip files in Semantically_Incongruent_or_Congruent_Eggplants_revised/stimuli
Extracting doi_10_5061_dryad_9ghx3ffkg__v20240921.zip...

Extraction complete. Files in dataset:
['README', 'N400Stimset_cloze-probability-survey_results.json', 'N400Stimset_cloze-probability-survey.json', 'N400Stimset_stimuli_parameters.tsv', 'N400Stimset_cloze-probability-survey_results.tsv', 'README.md', 'N400Stimset_cloze-probability-survey.tsv', 'stimuli.zip', 'N400Stimset_stimuli_parameters.json']


In [21]:
import os
import zipfile

dataset_path = 'dataset'
nested_zip = os.path.join(dataset_path, 'stimuli.zip')

if os.path.exists(nested_zip):
    print(f"Extracting nested archive: {nested_zip}...")
    with zipfile.ZipFile(nested_zip, 'r') as zip_ref:
        zip_ref.extractall(dataset_path)
    print("Extraction of nested stimuli complete.")
else:
    print("No nested stimuli.zip found.")

# Final verification of files
files = os.listdir(dataset_path)
print(f"\nFinal files in '{dataset_path}':")
for f in sorted(files):
    print(f" - {f}")

Extracting nested archive: dataset/stimuli.zip...
Extraction of nested stimuli complete.

Final files in 'dataset':
 - N400Stimset_cloze-probability-survey.json
 - N400Stimset_cloze-probability-survey.tsv
 - N400Stimset_cloze-probability-survey_results.json
 - N400Stimset_cloze-probability-survey_results.tsv
 - N400Stimset_stimuli_parameters.json
 - N400Stimset_stimuli_parameters.tsv
 - README
 - README.md
 - stimuli
 - stimuli.zip


In [23]:
import pandas as pd
import json

# Load TSV structure - this part usually works fine
tsv_path = 'dataset/N400Stimset_cloze-probability-survey_results.tsv'
df_tsv = pd.read_csv(tsv_path, sep='\t')

# Peek at the JSON file to diagnose the JSONDecodeError
json_path = 'dataset/N400Stimset_cloze-probability-survey_results.json'
print("--- JSON File Preview (First 5 lines) ---")
with open(json_path, 'r') as f:
    for _ in range(5):
        print(f.readline().strip())

print("\n--- TSV Columns ---")
print(df_tsv.columns.tolist())

display(df_tsv.head())

--- JSON File Preview (First 5 lines) ---
{
"#": {
"Description": "stimulus (sentence) order",
},


--- TSV Columns ---
['#', 'total_answers', 'correct_answers', 'cloze_probability', 'sentential_context', 'key']


,#,total_answers,correct_answers,cloze_probability,sentential_context,key
0,1,134,129,0.9627,On the sidewalk I draw with,chalk
1,2,134,123,0.9179,The pool was six feet,deep
2,3,132,118,0.8939,The diamond was not real it was,fake
3,4,133,131,0.9850,Down the sad woman's cheek came a,tear
4,5,134,134,1.0000,He likes his chicken deep,fried


In [26]:
import os

# Fix the script to use tab separator for the input file
script_path = 'Semantically_Incongruent_or_Congruent_Eggplants_revised/python/process_stimuli.py'
with open(script_path, 'r') as f:
    content = f.read()

# Replace the standard read_csv call with one that uses sep='\t'
fixed_content = content.replace("df = pd.read_csv(args.input)", "df = pd.read_csv(args.input, sep='\\t')")

with open(script_path, 'w') as f:
    f.write(fixed_content)

# Inspect line 172 of the parameters file to check for anomalies
params_file = 'dataset/N400Stimset_stimuli_parameters.tsv'
print(f"--- Inspecting {params_file} near line 172 ---")
with open(params_file, 'r') as f:
    lines = f.readlines()
    for i in range(170, 175):
        if i < len(lines):
            print(f"Line {i+1}: {repr(lines[i])}")

# Attempt to run the script again
print("\n--- Running the fixed script ---")
%cd Semantically_Incongruent_or_Congruent_Eggplants_revised/python
!python process_stimuli.py ../../dataset/N400Stimset_stimuli_parameters.tsv
%cd ../..

--- Inspecting dataset/N400Stimset_stimuli_parameters.tsv near line 172 ---
Line 171: 'sound170\tNPI_three(boy).wav\tI\tcount\tone\ttwo\tboy.\tn/a\tn/a\tn/a\tcount\t2.13415\t1.717623\t2.0627\t0.345077\t2\t4\t0.992424242\t1\tExpectation not disrupted/Congruent and incongruent type match\n'
Line 172: 'sound171\t(LD5_Eliminated)NPI_cute(fence).wav\tShe\tthinks\tall\tbaby\tanimals\tare\tfence.\tn/a\tcute\t2.815578\t2.277258\t2.773225\t0.495967\t2\t4\t0.953488372\t5\tDeleted for counting issue (expecting adjetive, and if noun needs to be plural) Hard to classify.\n'
Line 173: 'sound172\t(LD5_Eliminated)NPC_beard.wav\tSanta\thas\ta\tbig\twhite\tbeard.\tn/a\tn/a\tbeard\t2.613107\t2.184887\t2.563358\t0.378471\t2\t1\t0.976744186\t5\tCultural\n'
Line 174: 'sound173\tNPC_lawn.wav\tMy\tdad\tmows\tthe\tlawn.\tn/a\tn/a\tn/a\tgrass(lawn)\t2.305125\t1.833068\t2.273716\t0.440648\t2\t1\t0.887218045\t1\tExpectation not disrupted/Congruent and incongruent type match\n'
Line 175: 'sound174\t(LD5_Eliminated